# **II. News headlines**

For supplementary sentiment data and following literature on the topic, we scrape newspaper headlines. 

We begin by scraping for each firm of interest.

In [ ]:
from bs4 import BeautifulSoup
import pandas as pd
import requests
import time

# Specific URLs per theme
SOURCES = {
    "AAPL":     ("apple-inc",                                    39),
    "AMZN":    ("stream/84cf4073-a674-4a93-aef9-dcc1832a65cb", 33),
    "GOOG":  ("stream/6656b3e4-1a78-4960-90a0-8d422450f9a6", 11),
    "GOOGL":    ("stream/d6b12f0c-bf3f-4045-a07b-1e4e49103fd6", 32),
    "MSFT": ("stream/4f447b5d-53f5-41bd-ab42-3c0cfc161699", 25),
    "TSLA":     ("stream/35edec46-ef7b-4f9b-b85a-25174e6e07fa", 32),
}

def scrape_ft_topic(slug, start_page, ticker):
    articles = []
    page = start_page
    while True:
        url = f"https://www.ft.com/{slug}?page={page}"
        r = requests.get(url)
        soup = BeautifulSoup(r.text, "html.parser")
        items = soup.select(".stream-item[data-id]")

        if not items:
            print(f"\n  [{ticker}] page {page}: no items, stopping")
            break

        for item in items:
            title_tag = item.select_one(".o-teaser__heading a")
            li = item.find_parent("li")
            time_tag = li.find("time") if li else None
            if not title_tag:
                continue

            date = time_tag["datetime"][:10] if time_tag else ""
            title = title_tag.get_text(strip=True)

            if title.startswith("Lex."):
                title = title[len("Lex."):].strip()

            if date and date[:4] < "2015":
                print(f"\n  [{ticker}] p{page}: reached {date}, stopping")
                return articles

            if date and date[:4] < "2020":
                articles.append({"ticker": ticker, "date": date, "title": title})

        print(f"Scraping {ticker}: Page {page}, {len(articles)} articles", end="\r")
        page += 1
        time.sleep(1)

    return articles


all_articles = []
for ticker, (slug, start_page) in SOURCES.items():
    arts = scrape_ft_topic(slug, start_page, ticker)
    all_articles.extend(arts)

dfHL = pd.DataFrame(all_articles)
dfHL = dfHL.drop_duplicates(subset=["title", "date", "ticker"])
dfHL = dfHL.sort_values(["ticker", "date"], ascending=[True, False])

print(f"\nTotal: {len(dfHL)} articles")
print(dfHL.groupby("ticker").size())
dfHL.to_csv("data/output/HL_FinTimes_raw.csv", index=False)

Scraping AAPL: Page 150, 2776 articles
  [AAPL] p151: reached 2014-12-31, stopping
Scraping AMZN: Page 92, 1491 articles
  [AMZN] p93: reached 2014-12-29, stopping
Scraping GOOG: Page 87, 1912 articles
  [GOOG] p88: reached 2014-12-30, stopping
Scraping GOOGL: Page 53, 517 articles
  [GOOGL] page 54: no items, stopping
Scraping MSFT: Page 56, 793 articles
  [MSFT] p57: reached 2014-12-28, stopping
Scraping TSLA: Page 63, 782 articles
  [TSLA] p64: reached 2014-12-30, stopping

Total: 8307 articles
company
AAPL     2784
AMZN     1488
GOOG     1920
GOOGL     514
MSFT      810
TSLA      791
dtype: int64


Out of these 8308 FT articles, we only keep 3898 after keyword and manual cleaning: the FT thematic organization is poor, articles about apples (fruit) are fetched when using the page of the firm. As a solution, we also scrape Reuters, The Guardian, Bloomberg, The New York Times, The Wall Street Journal, CNBC, BBC\footnote{We made the choice not to leverage Yahoo Finance. Despite providing market-oriented news only, the quality of certain sources are questionable, often more oriented towards sensational day traders and not institutional quality news.}. Instead of designed advanced scrapers to counter the security norms of each website, we leverage Google search engine with the query: "intitle:{keyword} site:www.{site}.com after:2014-12-31 before:2020-01-01". The main limitation of this method is that headlines are sometimes truncated in Google results. 

We scrape the following keywords for each firm:
- AAPL: Apple, Iphone, Steve Jobs, Tim Cook, iPade, AirPods, MacBook
- AMZN: Amazon, Jeff Bezos, AWS
- GOOG/GOOGL: Google, Sundar Pichai, Alphabet
- MSFT: Satya Nadella, Microsoft
- TSLA: Tesla, Model 3, Elon Musk.

After cleaning (foreign languages, wrong thematic, customer-oriented articles, guides\footnote{We remove all articles mentioning apples (fruit), the Amazon(forest), and consumer-oriented artices identifiyed using keywords such as "How...", "review", "games", "which", "tips", "tricks", "best", "app", "guide", "opinion", "What...", "Why...", "How...", etc.}), we obtain a grand total of 11532 headlines. The main limitation of this method have a total of 11532 headlines. 